# Qwen Math GRPO/DAPO — 七组消融（Kaggle T4 一键运行）

从零手写的 GRPO 训练循环 + DAPO 四项单变量开关 + 熵正则扩展，在 **Qwen2.5-0.5B-Instruct + LoRA** 上跑 GSM8K 数学推理 RL。

## 使用方法（共 3 步）

1. Kaggle → **New Notebook** → File → **Import Notebook**，上传本文件
2. 右侧 Settings → Accelerator → **GPU T4 x1**（免费）
3. **Run All**，等跑完（七组 × 30 步 + 每组 pass@8 采样评测，约 5-8 小时，可随时回来查看进度）

> 想先快速验证？把 Cell 6 里的 `MAX_STEPS` 改成 `5`（约 40 分钟跑完七组）。

## 跑完产出（/kaggle/working/ 下，可下载）

- `ablation_results.json` — 每组最终指标（accuracy / format_rate / surprisal / clip_fraction / 零方差组占比 …）
- `ablation_summary.png` — 七组 reward 训练曲线 + 最终 accuracy 对比图
- 每步完整日志在 notebook 输出区（reward / KL / 熵 / hack rates 逐 step 可见）

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "请在右侧 Settings -> Accelerator 选择 GPU T4 后重新运行"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"   # Kaggle 服务器直连官方源

!rm -rf qwen-math-grpo-dapo
!git clone -q https://github.com/rayyy032/qwen-math-grpo-dapo.git
%cd qwen-math-grpo-dapo

!pip -q install "transformers>=4.45" datasets peft accelerate math-verify swanlab pyyaml tqdm
print("\n== 依赖安装完成 ==")

In [ ]:
# 27 项边界单测：boxed 提取 / 三级数值等价 / overlong 边界 / 端到端打分
!python -m pytest tests/ -q

In [ ]:
# ~1 分钟冒烟：验证 GPU 上整条流水线（left-padding 生成 -> reward -> loss -> 更新 -> eval）
!python scripts/train.py --algorithm dapo --dataset synthetic \
    --train-samples 12 --eval-samples 4 --max-steps 1 \
    --group-size 4 --groups-per-step 1 --max-new-tokens 48 \
    --device cuda --use-swanlab false

In [ ]:
# ============================================================
# 七组单变量消融（唯一需要看的配置区）
# ============================================================
PRESETS = ["grpo", "clip_higher", "dynamic", "token_level", "overlong", "dapo", "entropy_reg"]
MAX_STEPS = 30          # 每组步数；30 步约 25-50 min/组（dynamic 组会补采更慢）
DATASET = "gsm8k"       # gsm8k | gsm8k_math
TRAIN_SAMPLES = 500
EVAL_SAMPLES = 100
# ============================================================

import subprocess, time, json

REPO = "/kaggle/working/qwen-math-grpo-dapo"
results = {}

for algo in PRESETS:
    print(f"\n{'='*70}\n>>> preset: {algo}\n{'='*70}", flush=True)
    cmd = ["python", "scripts/train.py",
           "--algorithm", algo, "--dataset", DATASET,
           "--train-samples", str(TRAIN_SAMPLES),
           "--eval-samples", str(EVAL_SAMPLES),
           "--pass-at-k", "8", "--pass-k-samples", "50",
           "--max-steps", str(MAX_STEPS),
           "--device", "cuda", "--run-name", algo]
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        line = line.rstrip()
        print(line, flush=True)      # 实时进度
        lines.append(line)
    proc.wait()

    # ---- 解析该组结果（零正则，纯字符串拆分）----
    curve, final, eval_vals = [], {}, {}
    for line in lines:
        if line.startswith("[step"):
            body = line.split("] ", 1)[-1]
            vals = {}
            for part in body.split(" | "):
                k, _, v = part.partition("=")
                try:
                    vals[k] = float(v)
                except ValueError:
                    pass
            final = vals
            if "mean" in vals:          # reward/mean
                curve.append(vals["mean"])
        elif line.startswith("[eval]"):
            for part in line.split("] ", 1)[-1].split(" "):
                k, _, v = part.partition("=")
                if k == "accuracy" or k == "format_rate" or k.startswith("pass@"):
                    try:
                        eval_vals[k] = float(v)
                    except ValueError:
                        pass

    results[algo] = {
        "minutes": round((time.time() - t0) / 60, 1),
        "accuracy": eval_vals.get("accuracy"),
        "format_rate": eval_vals.get("format_rate"),
        "pass@8": eval_vals.get("pass@8"),
        "final_reward": final.get("mean"),
        "surprisal": final.get("mean_sampled_token_surprisal"),
        "clip_fraction": final.get("clip_fraction"),
        "zero_var_ratio": final.get("zero_variance_group_ratio"),
        "mean_completion_tokens": final.get("mean_completion_tokens"),
        "reward_curve": curve,
    }
    with open("/kaggle/working/ablation_results.json", "w") as f:   # 每组跑完即存，中断不丢
        json.dump(results, f, indent=2)
    print(f"\n[summary] {algo}: acc={eval_vals.get('accuracy')} pass@8={eval_vals.get('pass@8')} "
          f"({results[algo]['minutes']} min)", flush=True)

print("\nsaved -> /kaggle/working/ablation_results.json")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("/kaggle/working/ablation_results.json") as f:
    results = json.load(f)

df = pd.DataFrame(results).T
display(df[["accuracy", "pass@8", "format_rate", "final_reward", "surprisal",
            "clip_fraction", "zero_var_ratio", "mean_completion_tokens", "minutes"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for algo, r in results.items():
    if r.get("reward_curve"):
        axes[0].plot(r["reward_curve"], label=algo, alpha=0.85)
axes[0].set_xlabel("step")
axes[0].set_ylabel("mean reward")
axes[0].set_title("Training reward curves (7 presets)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

algos = list(results.keys())
accs = [results[a]["accuracy"] or 0 for a in algos]
colors = ["#9e9e9e"] * len(algos)
if "dapo" in algos:
    colors[algos.index("dapo")] = "#d62728"
if "entropy_reg" in algos:
    colors[algos.index("entropy_reg")] = "#1f77b4"
axes[1].bar(algos, accs, color=colors)
axes[1].set_ylabel("GSM8K accuracy")
axes[1].set_title("Final eval accuracy")
axes[1].tick_params(axis="x", rotation=30)
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/kaggle/working/ablation_summary.png", dpi=150)
print("saved -> /kaggle/working/ablation_summary.png")
plt.show()